
# Manejo de errores — Ingesta SharePoint → Bronze

## Propósito

Gestionar los errores producidos durante la ejecución de `ingest_sharepoint_bronze`, obteniendo el detalle técnico de la falla y actualizando el registro correspondiente en la tabla `etl_log`.

## Flujo

1. Obtener los parámetros de la ejecución fallida.
2. Consultar el detalle de la ejecución mediante Databricks Jobs API.
3. Extraer el detalle exacto de la excepción.
4. Actualizar el registro correspondiente en `etl_log`.
5. Registrar el estado `ERROR`, la fecha de finalización y el detalle del error.
6. Validar el registro actualizado.

## Task asociada

`ingest_sharepoint_bronze`

## Tabla de trazabilidad

`metadata_dataco.supply_chain.hd_etl_log`
## Resultado esperado

Actualizar el registro de `ingest_sharepoint_bronze` con:

* `fecha_fin`
* `estado = ERROR`
* `mensaje_error`

El mensaje debe conservar el detalle técnico de la excepción generada durante la ingesta.


In [0]:
# ============================================================
# 1. RECEPCIÓN DE PARÁMETROS Y RESOLUCIÓN DE LOG_ID
# ============================================================

import pyspark.sql.functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("error_code", "")
dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("result_state", "")

error_code = dbutils.widgets.get("error_code")
log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
result_state = dbutils.widgets.get("result_state")

TABLA_ETL_LOG = "metadata_dataco.supply_chain.hd_etl_log"
TASK_ORIGEN = "ingest_sharepoint_bronze"

# ============================================================
# FALLBACK DE LOG_ID
# ============================================================

if not log_id or log_id.strip() == "":
    
    print("⚠ log_id no recibido. Buscando ejecución EN_PROCESO...")

    registro_pendiente = (
        spark.table(TABLA_ETL_LOG)
        .filter(
            (F.col("notebook") == TASK_ORIGEN) &
            (F.col("estado") == "EN_PROCESO")
        )
        .orderBy(F.col("fecha_inicio").desc())
        .first()
    )

    if registro_pendiente:
        log_id = registro_pendiente["log_id"]
        print(f"✓ log_id recuperado: {log_id}")
    else:
        print("⚠ No se encontró registro EN_PROCESO.")

print(f"✓ Error code: {error_code}")
print(f"✓ Log ID: {log_id}")
print(f"✓ Run ID: {run_id}")
print(f"✓ Resultado: {result_state}")

In [0]:
# ============================================================
# 2. CONSULTA DEL DETALLE DEL TASK FALLIDO
# ============================================================

import requests

workspace_url = spark.conf.get(
    "spark.databricks.workspaceUrl"
)

token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

url = (
    f"https://{workspace_url}"
    "/api/2.2/jobs/runs/get-output"
)

response = requests.get(
    url,
    headers={
        "Authorization": f"Bearer {token}"
    },
    params={
        "run_id": run_id
    }
)

response.raise_for_status()

output_info = response.json()

print("✓ Detalle de ejecución consultado correctamente.")

In [0]:
# ============================================================
# 3. EXTRACCIÓN DEL DETALLE DEL ERROR
# ============================================================

error_detail = (
    output_info.get("error")
    or output_info.get("error_trace")
    or output_info.get("notebook_output", {}).get("result")
)

if not error_detail:
    error_detail = (
        "No fue posible obtener el detalle de la excepción "
        "mediante Databricks Jobs API."
    )

print("✓ Detalle del error:")
print(error_detail)

In [0]:
# ============================================================
# 4. ACTUALIZACIÓN DEL REGISTRO DE ERROR
# ============================================================

delta_log = DeltaTable.forName(
    spark,
    TABLA_ETL_LOG
)

mensaje_error = (
    f"Task: {TASK_ORIGEN} | "
    f"Error code: {error_code} | "
    f"Run ID: {run_id} | "
    f"Resultado: {result_state} | "
    f"Detalle: {error_detail}"
)

if log_id:

    delta_log.update(
        condition=F.col("log_id") == log_id,
        set={
            "fecha_fin": F.current_timestamp(),
            "estado": F.lit("ERROR"),
            "mensaje_error": F.lit(mensaje_error)
        }
    )

    print("✓ Registro actualizado correctamente.")
    print(f"✓ log_id: {log_id}")
    print("✓ Estado: ERROR")

else:

    print(
        "⚠ No se pudo actualizar etl_log porque "
        "no se encontró un log_id."
    )

In [0]:
# ============================================================
# 5. VALIDACIÓN DEL LOG DE ERROR
# ============================================================

if log_id:

    display(
        spark.table(TABLA_ETL_LOG)
        .filter(F.col("log_id") == log_id)
    )

In [0]:
%sql

-- ============================================================
-- VALIDACIÓN DEL ETL LOG
-- ============================================================

SELECT *
FROM metadata_dataco.supply_chain.hd_etl_log
ORDER BY fecha_inicio DESC;

In [0]:
# ============================================================
# ALERTA A SLACK
# ============================================================

import requests as slack_requests

try:
    webhook_url = dbutils.secrets.get(scope="alerts", key="slack_webhook_url")

    slack_payload = {
        "text": (
            f":rotating_light: *Fallo en pipeline DataCo*\n"
            f"*Task:* ingest_sharepoint_bronze\n"
            f"*Estado:* {result_state}\n"
            f"*Error code:* {error_code}\n"
            f"*Run ID:* {run_id}\n"
            f"*log_id:* {log_id}"
        )
    }

    slack_response = slack_requests.post(webhook_url, json=slack_payload, timeout=10)

    if slack_response.status_code == 200:
        print("✓ Alerta enviada a Slack")
    else:
        print(f"⚠️ Slack respondió con status {slack_response.status_code}")

except Exception as ex:
    print(f"⚠️ No se pudo enviar la alerta a Slack (scope/clave no configurado aún): {ex}")